# Ex2 - Getting and Knowing your Data

This time we are going to pull data directly from the internet.
Special thanks to: https://github.com/justmarkham for sharing the dataset and materials.

### Step 1. Import the necessary libraries

In [1]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Chipotle_Exercise').getOrCreate()

### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv).

### Step 3. Assign it to a variable called chipo.

In [67]:
import pandas as pd

url = 'https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv'
chipo = pd.read_csv(url, sep='\t')

chipo = spark.createDataFrame(chipo)


### Step 4. See the first 10 entries

In [12]:
chipo.show(10)

+--------+--------+--------------------+--------------------+----------+
|order_id|quantity|           item_name|  choice_description|item_price|
+--------+--------+--------------------+--------------------+----------+
|       1|       1|Chips and Fresh T...|                 NaN|    $2.39 |
|       1|       1|                Izze|        [Clementine]|    $3.39 |
|       1|       1|    Nantucket Nectar|             [Apple]|    $3.39 |
|       1|       1|Chips and Tomatil...|                 NaN|    $2.39 |
|       2|       2|        Chicken Bowl|[Tomatillo-Red Ch...|   $16.98 |
|       3|       1|        Chicken Bowl|[Fresh Tomato Sal...|   $10.98 |
|       3|       1|       Side of Chips|                 NaN|    $1.69 |
|       4|       1|       Steak Burrito|[Tomatillo Red Ch...|   $11.75 |
|       4|       1|    Steak Soft Tacos|[Tomatillo Green ...|    $9.25 |
|       5|       1|       Steak Burrito|[Fresh Tomato Sal...|    $9.25 |
+--------+--------+--------------------+-----------

### Step 5. What is the number of observations in the dataset?

In [13]:
# Solution 1
chipo.count()


4622

In [14]:
# Solution 2
chipo.printSchema()


root
 |-- order_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- item_name: string (nullable = true)
 |-- choice_description: string (nullable = true)
 |-- item_price: string (nullable = true)



### Step 6. What is the number of columns in the dataset?

In [15]:
len(chipo.columns)

5

### Step 7. Print the name of all the columns.

In [17]:
chipo.columns

['order_id', 'quantity', 'item_name', 'choice_description', 'item_price']

### Step 8. How is the dataset indexed?

In [42]:
'N/A'

'N/A'

### Step 9. Which was the most-ordered item?

In [20]:
chipo.groupBy('item_name').count().orderBy('count',ascending=False).show(1)

+------------+-----+
|   item_name|count|
+------------+-----+
|Chicken Bowl|  726|
+------------+-----+
only showing top 1 row


### Step 10. For the most-ordered item, how many items were ordered?

In [35]:
from pyspark.sql import functions as F

chipo.groupBy("item_name").agg(
    F.sum("quantity").alias("total_quantity"),
    F.count("item_name").alias("count")
  ).orderBy("total_quantity",ascending=False).show()

+--------------------+--------------+-----+
|           item_name|total_quantity|count|
+--------------------+--------------+-----+
|        Chicken Bowl|           761|  726|
|     Chicken Burrito|           591|  553|
| Chips and Guacamole|           506|  479|
|       Steak Burrito|           386|  368|
|   Canned Soft Drink|           351|  301|
|               Chips|           230|  211|
|          Steak Bowl|           221|  211|
|       Bottled Water|           211|  162|
|Chips and Fresh T...|           130|  110|
|         Canned Soda|           126|  104|
|  Chicken Salad Bowl|           123|  110|
|  Chicken Soft Tacos|           120|  115|
|       Side of Chips|           110|  101|
|      Veggie Burrito|            97|   95|
|    Barbacoa Burrito|            91|   91|
|         Veggie Bowl|            87|   85|
|       Carnitas Bowl|            71|   68|
|       Barbacoa Bowl|            66|   66|
|    Carnitas Burrito|            60|   59|
|    Steak Soft Tacos|          

### Step 11. What was the most ordered item in the choice_description column?

In [39]:
from pyspark.sql.functions import col
chipo.groupBy('choice_description').agg(
    F.sum('quantity').alias('total_quantity'),
    F.count('choice_description').alias('count')
).orderBy('total_quantity', ascending=False).filter(col("choice_description")!='NaN').show(1)

+------------------+--------------+-----+
|choice_description|total_quantity|count|
+------------------+--------------+-----+
|       [Diet Coke]|           159|  134|
+------------------+--------------+-----+
only showing top 1 row


### Step 12. How many items were orderd in total?

In [41]:
chipo.agg(F.sum('quantity').alias('total_quantity')).show()

+--------------+
|total_quantity|
+--------------+
|          4972|
+--------------+



### Step 13. Turn the item price into a float

#### Step 13.a. Check the item price type

In [44]:
chipo.schema['item_price'].dataType

StringType()

#### Step 13.b. Create a lambda function and change the type of item price

In [68]:
from pyspark.sql import functions as F
chipo = chipo.withColumn('item_price',
                 F.regexp_replace(F.col("item_price"),"^.",""))



In [70]:
chipo = chipo.withColumn('item_price',

                         col('item_price').cast("float")

                         )

#### Step 13.c. Check the item price type

In [71]:
chipo.schema['item_price'].dataType

FloatType()

### Step 14. How much was the revenue for the period in the dataset?

In [73]:
chipo.withColumn('revenue',
                 col('quantity')*col('item_price'))\
                 .agg(F.sum('revenue').alias('revenue'))\
      .withColumn('revenue',F.round('revenue',2)).show()

+--------+
| revenue|
+--------+
|39237.02|
+--------+



### Step 15. How many orders were made in the period?

In [79]:
chipo.agg(F.count_distinct("order_id").alias("contagem de pedidos")).collect()[0][0]

1834

### Step 16. What is the average revenue amount per order?

In [85]:
# Solution 1
chipo.withColumn('revenue',
                 col('quantity')*col('item_price')
                 )\
                 .agg(
                     F.sum('revenue').alias('revenue'),
                     F.count_distinct("order_id").alias("contagem"),


                  )\
                 .withColumn("mean",col("revenue")/col("contagem"))\
                 .select("mean")\
                 .show()



+-----------------+
|             mean|
+-----------------+
|21.39423104265914|
+-----------------+



### Step 17. How many different items are sold?

In [87]:
chipo.agg(
    F.count_distinct("item_name")
).show()

+-------------------------+
|count(DISTINCT item_name)|
+-------------------------+
|                       50|
+-------------------------+

